# 01 — Data acquisition and audit

Phase 1 of the Cape Town service-request resolution classifier.

This notebook prepares the project and, when a raw file is present, describes that file. It does not clean the extract, fit a preprocessor, or train a model.

Run it from the top after placing the original download in `data/raw/`. If the file is missing, the notebook stops at the manual download steps. It does not create a substitute dataset.


## 1. Problem definition

When a City of Cape Town municipal service request is logged, service departments need an early view of how it is likely to end. The classifier predicts one of three outcomes:

- **Resolved Fast**
- **Resolved Slow**
- **Unresolved**

The prediction must use only information that is known at logging. Later notebooks will compare models with macro-F1 as the primary metric, and they will pay particular attention to recall for Resolved Slow and Unresolved. Those models are out of scope here.


## 2. Dataset

The project plan names this source:

- **Owner:** City of Cape Town
- **Portal:** City of Cape Town Open Data Portal
- **Dataset:** SAP C3 Notifications
- **Collection period in the plan:** 2023 until 18 September 2026

The exact download URL, the licence text, and the download date are **not in this workspace**. They are recorded as missing in `reports/data_source.md`. This notebook does not guess a URL and does not download anything.

The raw file, once obtained, stays unchanged in `data/raw/`.


## 3. Leakage and the observation window

The label will later be built from **Created On Date** and **Completed Date**. **Completed Date** is not a model feature. **Changed on** can reflect updates after logging, so it is excluded as well.

An open request that was logged only a few days before the end of the study can look Unresolved simply because not enough time has passed. Those rows need an observation window. The fast/slow boundary also has to be chosen from the resolution-time distribution.

Both thresholds are unset in Phase 1 (`None` in `src/config.py`). This notebook does not choose them and does not build the final label column.


## 4. Setup

The next cell finds the project root, imports the shared modules, and sets the random seed. PyTorch is imported only to confirm the chosen neural-network framework loads. No network is defined or trained.


In [1]:
from pathlib import Path
import sys

import matplotlib
import pandas as pd
import torch

# Headless execution can save figures later. JupyterLab still displays outputs.
matplotlib.use("Agg", force=False)

cwd = Path.cwd().resolve()
project_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src" / "config.py").is_file():
        project_root = candidate
        break
if project_root is None:
    raise FileNotFoundError(
        "Could not find src/config.py. Start JupyterLab from the project folder."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config
from src.data_loader import (
    RawDataNotFoundError,
    audit_frame,
    list_raw_files,
    load_raw_table,
    manual_download_instructions,
    resolve_raw_file,
)
from src.preprocessing import planned_column_report, select_model_features

config.set_random_seeds(config.RANDOM_SEED)
print("Project root:", project_root)
print("Random seed:", config.RANDOM_SEED)
print("PyTorch:", torch.__version__)
print("pandas:", pd.__version__)
print("Target labels:", config.TARGET_LABELS)
print("Fast/slow threshold (days):", config.FAST_SLOW_THRESHOLD_DAYS)
print("Observation window (days):", config.OBSERVATION_WINDOW_DAYS)
print("Source page URL:", config.SOURCE_PAGE_URL)
print("Planned collection period:", config.PLANNED_COLLECTION_PERIOD)


Project root: /workspace
Random seed: 42
PyTorch: 2.14.1+cpu
pandas: 3.0.6
Target labels: ('Resolved Fast', 'Resolved Slow', 'Unresolved')
Fast/slow threshold (days): None
Observation window (days): None
Source page URL: None
Planned collection period: 2023 until 18 September 2026


## 5. Project folders

These paths are where later notebooks will read and write. This cell checks that the folders exist. It does not create data.


In [2]:
for path in config.required_directories():
    print(f"{'ok' if path.is_dir() else 'MISSING':7}  {path.relative_to(config.PROJECT_ROOT)}")

print("Raw files:", [path.name for path in list_raw_files()])


ok       data/raw
ok       data/interim
ok       data/processed
ok       notebooks
ok       models
ok       app
ok       results/figures
ok       results/tables
ok       reports
ok       tests
ok       src
Raw files: []


## 6. Locate the raw extract

If `data/raw/` is empty, the cell prints the manual steps and sets `data_available` to false. No sample rows are invented. If several files are present, the cell also stops, because the loader will not choose one by itself.


In [3]:
data_available = False
raw_path = None
try:
    raw_path = resolve_raw_file()
except (RawDataNotFoundError, ValueError) as error:
    print(error)
else:
    data_available = True
    print("Raw file found:", raw_path)
    print("The file is read only. This notebook does not overwrite it.")


No raw City of Cape Town extract is available to this project yet.

The project plan names the dataset, but this workspace does not contain an
exact download URL or a working download mechanism. Do not invent one.

1. Open the City of Cape Town Open Data Portal.
2. Search for SAP C3 Notifications: service requests from 2023 until 18 September 2026.
3. On the dataset page, confirm the owner and the collection period before downloading.
4. Copy the page URL, the licence or terms shown there, and the download date into reports/data_source.md.
5. Save the original file, unchanged, in data/raw/.
6. If you download more than one file, set DATA_RAW_FILE in .env to the file this audit should read.
7. Do not commit the raw file. data/raw/ is gitignored except for .gitkeep.

Then re-run the audit from the top.


## 7. Audit the file that was actually loaded

The next cell runs only when a raw file was found. It prints the row count, column names, dtypes, and missing counts from that file, and it checks the three column names given in the project brief.

A missing planned name means the extract uses different wording, or that the file is not the expected dataset. The notebook does not rename columns to force a match.

When no file is loaded, the cell prints a skip message. That message is not an audit result.


In [4]:
if not data_available:
    print("Audit skipped. No raw extract is loaded, so there are no column names or counts to report.")
    print("After the file is saved in data/raw/, run this notebook again from the top.")
else:
    frame = load_raw_table(raw_path)
    summary = audit_frame(frame)
    print(f"Rows: {summary['row_count']}")
    print(f"Columns: {summary['column_count']}")
    column_table = pd.DataFrame(
        {
            "column": summary["columns"],
            "dtype": [summary["dtypes"][name] for name in summary["columns"]],
            "missing_count": [summary["missing_count"][name] for name in summary["columns"]],
            "missing_percent": [round(summary["missing_percent"][name], 2) for name in summary["columns"]],
        }
    )
    display(column_table)
    name_report = planned_column_report(frame.columns)
    display(name_report)
    kept = select_model_features(frame.columns)
    removed = [name for name in frame.columns if name not in kept]
    print("Columns removed by the current leakage list:")
    print(removed if removed else "(none of the known leakage names were found)")
    print(
        "Columns still in the feature list still need a manual leakage review. "
        "This notebook does not drop them automatically."
    )
    table_path = config.TABLES_DIR / "01_column_audit.csv"
    column_table.to_csv(table_path, index=False)
    print("Wrote", table_path.relative_to(config.PROJECT_ROOT))


Audit skipped. No raw extract is loaded, so there are no column names or counts to report.
After the file is saved in data/raw/, run this notebook again from the top.


## 8. What this phase does not do

- It does not estimate the fast/slow threshold or the observation window.
- It does not impute, encode, scale, or split the data.
- It does not fit a dummy baseline, a logistic regression, a tree, or a neural network.
- It does not fill `reports/data_source.md` with a URL, a licence, or a download date that was not on the portal page.

## Next step

Download the SAP C3 Notifications extract using the steps in `reports/data_source.md`. Record the real URL, licence, and download date there. Re-run this notebook and use the printed column list before writing any dataset-specific cleaning. The following notebook, exploratory data analysis, is not created yet.
